## Updated CLI commands

This notebook was updated to remove dependency on `hera.utils.data.cli_toolkit_repository`.

Use the new CLIs instead:
- `hera-toolkit import-json --project <PROJECT> --file <REPOSITORY_JSON_PATH>`
- `hera-toolkit register --project <PROJECT> --cls <PYTHON_CLASSPATH> --name <DS_NAME>`
- `hera-project repository load <REPOSITORY_NAME> <PROJECT> [--overwrite]`

> Replace placeholders like `<PROJECT>` and `<REPOSITORY_JSON_PATH>` with your values.


# Hera Dynamic Toolkits — End‑to‑End Tutorial

This notebook shows **how to register, load, and use dynamic toolkits** in Hera using the new `Class` data type and the Toolkit Repository.
It includes a **quick refresher of `DataHandler_Class`**, a **clean CLI flow to register a toolkit**, and a **concrete IMS example**.


## What you will learn

- What the `Class` data type is and how `DataHandler_Class.getData(...)` works.
- How to **register a toolkit** as a datasource (DB document).
- How to **load a toolkit dynamically** (from DB or static registry).
- A **practical IMS walkthrough** (using your own API token).
- Troubleshooting common issues.


## Prerequisites

- You have the Hera repository on your machine (this notebook is meant to be run inside your Hera venv).
- MongoDB is running and Hera can read/write measurement documents.
- (For IMS only) You have a valid **IMS API token**.
- Optional dependencies for some experiments (e.g., `argos`) are installed if needed by that experiment.


## 0) Verify basic environment

In [ ]:
# You can run this cell safely.
import sys, importlib, json, os, pathlib

print("Python:", sys.version)
for mod in ["hera", "mongoengine"]:
    try:
        __import__(mod)
        print(f"✅ import {mod} OK")
    except Exception as e:
        print(f"❌ import {mod} FAILED -> {e}")

## 1) Quick refresher: `DataHandler_Class`

The `Class` datatype lets Hera load a class dynamically from code on disk.

**Inputs**
- `resource` — a *folder path* that contains the module you want to import (it is added to `sys.path`).
- `desc` — a dict with:
  - `classpath` (**required**): full import path like `pkg.module.ClassName`.
  - `parameters`/**`params`** (optional): constructor kwargs.
  - `instantiate` (optional, default `True`): if `False` return the **class** (type) instead of an **instance**.

**Parameter merge rule** (we use *Option B*): if a key appears both in `desc["parameters"]` and in function `**kwargs`, **the value in `desc["parameters"]` wins** (overrides).


In [ ]:
# Minimal example: create a tiny package on disk and load it via DataHandler_Class
from hera.datalayer.datahandler import DataHandler_Class
import pathlib, textwrap

# Write a one-file package under /tmp
root = pathlib.Path("/tmp/class_demo_pkg")
(root / "mypkg").mkdir(parents=True, exist_ok=True)
(root / "mypkg" / "__init__.py").write_text("")
(root / "mypkg" / "mymod.py").write_text(textwrap.dedent("""
class TestClass:
    def __init__(self, a=0, b="ok", **kw):
        self.a, self.b, self.kw = a, b, kw
    def snapshot(self):
        return {"a": self.a, "b": self.b, "kw": self.kw}
    def __repr__(self):
        return f"<TestClass a={self.a} b={self.b}>"
"""))

resource_dir = str(root)            # points at folder that contains 'mypkg'
classpath   = "mypkg.mymod.TestClass"

# 1) Return an INSTANCE (default instantiate=True)
obj = DataHandler_Class.getData(
    resource=resource_dir,
    desc={"classpath": classpath, "parameters": {"a": 1, "b": "hello"}}
)
print("Instance:", obj, "snapshot:", obj.snapshot())

# 2) Return the CLASS object (instantiate=False)
cls = DataHandler_Class.getData(
    resource=resource_dir,
    desc={"classpath": classpath, "instantiate": False}
)
print("Class object:", cls, "| is type?", isinstance(cls, type))

## 2) Register a toolkit in the Toolkit Repository (DB)

You can register *any* toolkit class (internal, experiment, or external) as a **datasource** so Hera can load it dynamically later.

### Option A — CLI (recommended)
Run this in a terminal from your Hera repo root:


In [ ]:
# (Shell example) Replace placeholders with your real values.
# hera-toolkit import-json --project <PROJECT> --file <REPOSITORY_JSON_PATH> register-datasource \
#   --project <YourProject> \
#   --name <ToolkitShortName> \
#   --classpath <pkg.module.ClassName> \
#   --resource <path/to/folder/that/contains/the/package> \
#   --params '{"key1": "value1", "key2": 123}' \
#   --version 0.0.1 \
#   --overwrite

# Example from the unit tests (for reference only):
# hera-toolkit import-json --project <PROJECT> --file <REPOSITORY_JSON_PATH> register-datasource #   --project UnitTestProject #   --name DemoToolkit_DS #   --classpath mypkg.mytoolkit.DemoToolkit #   --resource /tmp/rtk_demo #   --params '{"alpha": 7}' #   --version 0.0.1 #   --overwrite


### Option B — Python API (`ToolkitHome.registerToolkit`)

If you already have the class object in memory, you can register it programmatically (handy in scripts).

In [ ]:
# Example (in-memory class written to disk) — you can run this cell as-is.
import pathlib, textwrap
from hera.toolkit import ToolkitHome

# Write TinyToolkit to a temp directory so registerToolkit can find it by path
_tk_root = pathlib.Path("/tmp/tinytoolkit_demo")
_tk_root.mkdir(parents=True, exist_ok=True)
(_tk_root / "TinyToolkit.py").write_text(textwrap.dedent("""
class TinyToolkit:
    def __init__(self, projectName=None, filesDirectory=None, alpha=42, **kw):
        self.projectName = projectName
        self.filesDirectory = filesDirectory
        self.alpha = alpha
    def ping(self): return f"alpha={self.alpha}"
"""))

th = ToolkitHome()
doc = th.registerToolkit(
    toolkit_name="TinyToolkit",
    toolkit_path=str(_tk_root),
    params={"alpha": 99},        # constructor kwargs to remember in the datasource
    version=(0, 0, 1),
    overwrite=True,
)
print("Registered datasource:", doc.resource, doc.desc)

## 3) Discover and load toolkits

- **List toolkits** (static + dynamic) for a project.
- **Load** by name using `ToolkitHome.getToolkit(...)` (if present in static registry or DB).
- Or **load directly** via the datasource document using `DataHandler_Class.getData(...)`.


In [ ]:
from hera.toolkit import ToolkitHome
from hera.datalayer import Project
import sys, importlib

# List all toolkits (static built-ins + any dynamically registered ones)
th = ToolkitHome()
df = th.getToolkitTable()
print(df.head(20))

# Load via DB document — toolkit registered with registerToolkit uses dataFormat="string"
# so the path is the resource; we import it directly from the filesystem.
p = Project(projectName="defaultProject")
docs = p.getMeasurementsDocuments(type="ToolkitDataSource", datasourceName="TinyToolkit")
if docs:
    doc = docs[0]
    tk_path = doc.resource
    tk_name = doc.desc.get("datasourceName", "TinyToolkit")
    if tk_path not in sys.path:
        sys.path.insert(0, tk_path)
    mod = importlib.import_module(tk_name)
    cls = getattr(mod, tk_name)
    params = doc.desc.get("params", {})
    obj = cls(**params)
    print("Loaded instance:", obj, "| ping:", obj.ping())
else:
    print("No 'TinyToolkit' registered — run the register step above first.")

## 4) IMS — full example

This section shows how to load the **IMS experiment** dynamically and verify your access.

### 4.1 Create `token.json`

Create a file at `~/hera-ims/token.json` with **your own token**:

```json
{
  "Authorization": "ApiToken <YOUR_IMS_TOKEN_HERE>"
}
```

> Keep this file **private**. Do not commit it to Git.


### 4.2 Register IMS as a datasource (CLI)

Run from your Hera repo root (adjust paths if your `hera-ims` lives elsewhere):

```bash
hera-toolkit import-json --project <PROJECT> --file <REPOSITORY_JSON_PATH> register-datasource   --project UnitTestProject   --name IMS   --classpath IMS_experiment.IMS_experiment   --resource "$HOME/hera-ims/code"   --params '{"projectName":"UnitTestProject","pathToExperiment":"'$HOME'/hera-ims","filesDirectory":"'$HOME'/hera-ims/data"}'   --version 0.0.1   --overwrite
```

This creates a **ToolkitDataSource** document named `IMS`.


### 4.3 Load IMS and perform a light check

We import IMS via the `Class` handler and do a small non-destructive check (no heavy downloads).


In [ ]:
# IMS section: only runs if the IMS datasource and token.json are present.
# Skip gracefully if prerequisites are missing.
from hera.datalayer import Project
from hera.datalayer.datahandler import DataHandler_Class
import os, json

project = "UnitTestProject"
p = Project(projectName=project)
docs = p.getMeasurementsDocuments(type="ToolkitDataSource", datasourceName="IMS")

if not docs:
    print("IMS datasource not found. Run the CLI register command first. (Skipping IMS section.)")
else:
    doc = docs[0]
    ims = DataHandler_Class.getData(resource=doc.resource, desc=doc.desc)
    print("Loaded IMS object:", ims)

    tok_path = os.path.expanduser("~/hera-ims/token.json")
    if not os.path.isfile(tok_path):
        print("Missing ~/hera-ims/token.json — create it with your API token. (Skipping live check.)")
    else:
        import requests
        token = json.load(open(tok_path))
        url = "https://api.ims.gov.il/v1/Envista/stations"
        r = requests.get(url, headers={"Authorization": token["Authorization"]}, timeout=30)
        print("IMS stations HTTP:", r.status_code)
        stations = r.json()
        print("Received items:", len(stations) if isinstance(stations, list) else type(stations))
        print("Sample:", (stations[:2] if isinstance(stations, list) else stations))

## 5) Troubleshooting

- **`ModuleNotFoundError`** for your toolkit classpath  
  Ensure the **`resource`** you registered points to a folder that contains the module or its **parent** folder.  
  The handler adds it to `sys.path` before doing `import <module>`.

- **`token.json` missing** (IMS)  
  Create `~/hera-ims/token.json` with your **Authorization** header, e.g. `{"Authorization":"ApiToken ... "}`.

- **`argos` missing** (experiments that depend on it)  
  If the experiment requires `argos`, install it or use a stub that matches your use case.

- **Version mismatches**  
  If installing ARGOS or other dependencies changed package versions, re-run `pip check` and adjust env as needed.

- **Default project config error**  
  If you see `Default project cannot use configuration`, instantiate your toolkits with a **real project name** (e.g., `UnitTestProject`).

- **Duplicated registration**  
  Use `--overwrite` in the CLI or pass `overwrite=True` to the `registerToolkit(...)` API.


## Wrap‑up

- **Dynamic registration** lets Hera discover new toolkits without editing a central Python dict.
- The **`Class`** datatype is the glue: it records where the code lives and how to import it.
- Use the **CLI** to register and list, and load via `ToolkitHome` or `DataHandler_Class`.

You're ready to add more experiments/toolkits and load them on demand ✨.
